# harness

> The agentic loop: skill discovery and loading, fenced python execution against the persistent
> executor, and automatic controlled compaction -- all over a rishi `Chat`.

In [ ]:
#| default_exp harness

In [ ]:
#| hide
from nbdev.showdoc import *

One code channel instead of many JSON tools: small on-device models follow a single
\`\`\`python-fence convention far more reliably than a wide tool schema (and litert's
arguments-as-floats quirk disappears). Each turn the harness extracts the reply's last fence, runs
it through the `Executor` (approval-gated, persistent namespace), and feeds the output back as a
\`\`\`result fence until the model answers in prose, `done(harness)` is true, or `max_rounds` is hit.

Skills picked by the `Finder` are loaded lazily: SKILL.md bodies go into the message as `<skill>`
blocks; pyskills are star-imported into the executor namespace *and* documented in a `<skill>` block.
After each task the harness checks `needs_compact` and swaps in a compacted chat when the context
crosses the threshold.

In [ ]:
#| export
import re
from fastcore.utils import store_attr
from ramabana.registry import Registry
from ramabana.finder import Finder
from ramabana.executor import Executor
from ramabana.compact import compact, needs_compact

In [ ]:
#| export
_FENCE = re.compile(r'```(?:python|py)[ \t]*\n(.*?)```', re.S)

def extract_fence(txt:str):
    'The last ```python fence in `txt`, or None.'
    m = _FENCE.findall(txt or '')
    return m[-1].strip() if m else None

def _text(r):
    if isinstance(r, str): return r
    try:
        from rishi.core import resp_text
        return resp_text(r)
    except ImportError: return str(r)

In [ ]:
#| export
_SP = '''You are a coding agent running on a local model. Solve tasks by writing python.

Reply with a single ```python fence to run code; the harness executes it in a persistent namespace
and sends the output back in a ```result fence. Variables survive between fences and between tasks.
When the task is done, answer in prose with no fence.

Skills extend what you can do. When a skill is loaded, its docs appear in a <skill> block; import
and call it from python fences as those docs describe.

Available skills:
{catalog}'''

In [ ]:
#| export
class Harness:
    'Agentic loop over a rishi `Chat`: skill finding + loading, fenced execution, controlled compaction.'
    def __init__(self,
                 chat=None,          # rishi Chat (or anything callable returning text); lazily built if None
                 registry=None,      # skill Registry (default: discover from cwd + entry points)
                 finder=None,        # Finder over the registry (default: lexical)
                 executor=None,      # Executor for python fences (default: approval-gated fresh namespace)
                 approve=None,       # rishi-style approve(tool_call)->bool for the default executor/chat
                 retain=(),          # information-to-retain instructions used at every compaction
                 compact_at:float=0.85, # pct_full threshold that triggers compaction after a task
                 keep_skills:int=2,  # max skills auto-loaded per task
                 max_rounds:int=6,   # cap on execute/respond rounds per task
                 done=None,          # done(harness)->bool early-stop check after each round
                 chat_kw:dict=None): # extra kwargs for the lazily-built rishi Chat
        self.registry = registry if registry is not None else Registry()
        self.finder = finder if finder is not None else Finder(self.registry)
        self.executor = executor if executor is not None else Executor(approve=approve)
        store_attr('approve,retain,compact_at,keep_skills,max_rounds,done,chat_kw')
        self._chat, self.loaded = chat, {}
    @property
    def chat(self):
        if self._chat is None:
            from rishi.core import Chat
            self._chat = Chat(sp=self.sys_prompt(), approve=self.approve, **(self.chat_kw or {}))
        return self._chat
    def sys_prompt(self): return _SP.format(catalog=self.registry.catalog() or '(none)')
    def load_skill(self, skill):
        'Load one skill: pyskills are star-imported into the executor; returns the `<skill>` context block.'
        s = self.registry.get(skill) if isinstance(skill, str) else skill
        if s is None or s.name in self.loaded: return ''
        ctx = self.registry.load(s)
        if s.kind == 'py': self.executor(f'from {s.source} import *')
        self.loaded[s.name] = s
        return f'<skill name="{s.name}">\n{ctx}\n</skill>'
    def load_skills(self, request:str):
        'Pick skills for `request` with the finder and load the new ones; returns their joined context.'
        return '\n\n'.join(filter(None, (self.load_skill(s) for s in self.finder.pick(request, k=self.keep_skills))))
    def __call__(self, task:str):
        'Run one task to completion; returns the final model response.'
        ctx = self.load_skills(task)
        r = self.chat(f'{ctx}\n\n{task}' if ctx else task)
        for _ in range(self.max_rounds):
            code = extract_fence(_text(r))
            if code is None: break
            out = self.executor(code).rstrip('\n')
            r = self.chat(f'```result\n{out}\n```')
            if self.done is not None and self.done(self): break
        if needs_compact(self.chat, self.compact_at): self.compact()
        return r
    def compact(self, **kw):
        'Compress the conversation now (retaining `self.retain`); swaps in a fresh chat on the same engine.'
        self._chat = compact(self.chat, retain=self.retain, **kw)
        return self._chat

In [ ]:
assert extract_fence('do this:\n```python\nx=1\n```\nok') == 'x=1'
assert extract_fence('```py\na\n```\ntext\n```python\nb\n```') == 'b'   # last fence wins
assert extract_fence('no code here') is None and extract_fence(None) is None

In [ ]:
import tempfile
from pathlib import Path
tmp = tempfile.mkdtemp()
d = Path(tmp)/'.claude/skills/math'; d.mkdir(parents=True)
(d/'SKILL.md').write_text('---\nname: math\ndescription: arithmetic helpers for math questions\n---\nJust use python arithmetic.')

class _ScriptedChat:
    'Returns canned replies; records what it was sent.'
    def __init__(self, replies): self.replies, self.sent, self.hist = list(replies), [], []
    def __call__(self, msg):
        self.sent.append(msg)
        return self.replies.pop(0)

chat = _ScriptedChat(['I will compute it.\n```python\nans = 6*7\nprint(ans)\n```',
                      'The answer is 42.'])
h = Harness(chat=chat, registry=Registry(root=tmp, include_py=False), max_rounds=3)
res = h("what's 6*7? this is a math question about arithmetic")
assert res == 'The answer is 42.'
assert '<skill name="math">' in chat.sent[0]   # skill body was injected into the task message
assert '```result\n42\n```' in chat.sent[1]      # executor output fed back as a result fence
assert h.executor.ns['ans'] == 42                  # namespace persists after the task
assert 'math' in h.loaded

In [ ]:
# denial path: executor gate feeds the refusal back to the model instead of running the code
chat2 = _ScriptedChat(['```python\nx=1\n```', 'ok, stopped.'])
h2 = Harness(chat=chat2, registry=Registry(root=tmp, include_py=False),
             approve=lambda tc: False, max_rounds=2)
h2('run something')
assert '```result\nDenied by human operator\n```' in chat2.sent[1]

In [ ]:
# max_rounds caps a model that never stops emitting fences
chat3 = _ScriptedChat(['```python\n1\n```']*5)
h3 = Harness(chat=chat3, registry=Registry(root=tmp, include_py=False), max_rounds=2)
h3('loop forever')
assert len(chat3.sent) == 3   # task + 2 result rounds, then the cap

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()